In [ ]:
import csv
import random
import uuid
import time
from datetime import datetime, timedelta

# ==========================================
# 1. CẤU HÌNH THÔNG SỐ
# ==========================================
TOTAL_ORDERS = 5_000_000
BUFFER_SIZE = 50_000

START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2025, 12, 31)
TOTAL_SECONDS = int((END_DATE - START_DATE).total_seconds())

STATUS_CHOICES =['delivered', 'shipped', 'canceled', 'invoiced', 'processing']
STATUS_WEIGHTS =[97.0, 1.2, 0.6, 0.6, 0.6]

# --- ĐIỀU CHỈNH ĐỂ ĐẠT > 10 TRIỆU ORDER ITEMS ---
# Trọng số mới: 40% mua 1 món, 30% mua 2 món, 15% mua 3 món, 10% mua 4 món, 5% mua 5 món.
# Kỳ vọng (Expected Value) = 1*0.4 + 2*0.3 + 3*0.15 + 4*0.1 + 5*0.05 = 2.1 items / order.
# 5,000,000 orders * 2.1 = ~10,500,000 items.
ITEMS_PER_ORDER_CHOICES =[1, 2, 3, 4, 5]
ITEMS_PER_ORDER_WEIGHTS =[40.0, 30.0, 15.0, 10.0, 5.0]

# ==========================================
# 2. LOAD DỮ LIỆU GỐC TỪ FILE DB
# ==========================================
print("Đang Load Customer_IDs và Product_IDs...")

def load_ids(filename, fake_prefix, count):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            return[line.strip().split(',')[0] for line in f if line.strip() and "id" not in line.lower()]
    except FileNotFoundError:
        print(f"⚠️ Không tìm thấy {filename}. Sẽ dùng ID giả lập ({fake_prefix})...")
        return[f"{fake_prefix}_{i}" for i in range(1, count + 1)]

customer_ids = load_ids('customer_ids.csv', 'CUST', 100000)
product_ids = load_ids('product_ids.csv', 'PROD', 32000)

# ==========================================
# 3. THUẬT TOÁN SINH DỮ LIỆU
# ==========================================
print(f"Bắt đầu sinh {TOTAL_ORDERS:,} Orders chuẩn thực tế...")
start_time = time.time()

with open('mock_orders.csv', 'w', newline='', encoding='utf-8') as f_orders, \
     open('mock_order_items.csv', 'w', newline='', encoding='utf-8') as f_items:

    writer_orders = csv.writer(f_orders)
    writer_items = csv.writer(f_items)

    writer_orders.writerow(['order_id', 'customer_id', 'status', 'order_purchase_timestamp'])
    writer_items.writerow(['order_id', 'order_item_id', 'product_id', 'price', 'freight_value'])

    orders_buffer, items_buffer = [],[]
    total_items = 0

    for i in range(1, TOTAL_ORDERS + 1):
        # --- BẢNG ORDERS ---
        order_id = uuid.uuid4().hex
        customer_id = random.choice(customer_ids)
        status = random.choices(STATUS_CHOICES, weights=STATUS_WEIGHTS, k=1)[0]

        random_seconds = random.randint(0, TOTAL_SECONDS)
        order_date = START_DATE + timedelta(seconds=random_seconds)
        order_purchase_timestamp = order_date.strftime('%Y-%m-%d %H:%M:%S')

        orders_buffer.append([order_id, customer_id, status, order_purchase_timestamp])

        # --- BẢNG ORDER_ITEMS ---
        num_items = random.choices(ITEMS_PER_ORDER_CHOICES, weights=ITEMS_PER_ORDER_WEIGHTS, k=1)[0]
        total_items += num_items

        for item_id in range(1, num_items + 1):
            product_id = random.choice(product_ids)

            price = random.lognormvariate(mu=4.3, sigma=0.8)
            price = round(min(max(price, 5.0), 4000.0), 2)

            freight_value = random.lognormvariate(mu=2.8, sigma=0.6)
            freight_value = round(min(max(freight_value, 0.0), 300.0), 2)

            items_buffer.append([order_id, item_id, product_id, price, freight_value])

        # Xả Buffer
        if i % BUFFER_SIZE == 0:
            writer_orders.writerows(orders_buffer)
            writer_items.writerows(items_buffer)
            orders_buffer.clear()
            items_buffer.clear()
            print(f" Đã sinh: {i:,} / {TOTAL_ORDERS:,} Orders... (Đã tạo {total_items:,} Items)")

    # Xả nốt phần còn lại
    if orders_buffer:
        writer_orders.writerows(orders_buffer)
        writer_items.writerows(items_buffer)

end_time = time.time()
print("\n" + "="*50)
print(f"🎉 HOÀN TẤT! Đã sinh {TOTAL_ORDERS:,} Orders và {total_items:,} Items.")
print(f"Thời gian: {round(end_time - start_time, 2)} giây")
print("="*50)

Đang Load Customer_IDs và Product_IDs...
Bắt đầu sinh 5,000,000 Orders chuẩn thực tế...
 Đã sinh: 50,000 / 5,000,000 Orders... (Đã tạo 105,142 Items)
 Đã sinh: 100,000 / 5,000,000 Orders... (Đã tạo 209,950 Items)
 Đã sinh: 150,000 / 5,000,000 Orders... (Đã tạo 314,859 Items)
 Đã sinh: 200,000 / 5,000,000 Orders... (Đã tạo 419,648 Items)
 Đã sinh: 250,000 / 5,000,000 Orders... (Đã tạo 524,492 Items)
 Đã sinh: 300,000 / 5,000,000 Orders... (Đã tạo 628,842 Items)
 Đã sinh: 350,000 / 5,000,000 Orders... (Đã tạo 734,031 Items)
 Đã sinh: 400,000 / 5,000,000 Orders... (Đã tạo 838,874 Items)
 Đã sinh: 450,000 / 5,000,000 Orders... (Đã tạo 943,406 Items)
 Đã sinh: 500,000 / 5,000,000 Orders... (Đã tạo 1,048,280 Items)
 Đã sinh: 550,000 / 5,000,000 Orders... (Đã tạo 1,153,219 Items)
 Đã sinh: 600,000 / 5,000,000 Orders... (Đã tạo 1,258,272 Items)
 Đã sinh: 650,000 / 5,000,000 Orders... (Đã tạo 1,363,768 Items)
 Đã sinh: 700,000 / 5,000,000 Orders... (Đã tạo 1,468,934 Items)
 Đã sinh: 750,000 / 5